# Mundial 2026 — Notebook de DATOS

Construye y mantiene los ficheros de `data/` que consume `mundial2026_modelo.ipynb`.
Este notebook **sí toca la red** (Kaggle, Transfermarkt y, opcionalmente, FBref);
el notebook de modelo no hace ninguna petición.

**Salidas (el "contrato" con el notebook de modelo):**

| Fichero | Contenido | Fuente |
|---|---|---|
| `data/world_cup_matches.csv` | Partidos del Mundial: marcador, xG, posesión, tiros, córners, faltas, fueras de juego, paradas, tarjetas | Kaggle (+ FBref como complemento opcional) |
| `data/pre_world_cup_form.csv` | Últimos 5 partidos pre-Mundial de las 48 selecciones + `competition_weight` | Transfermarkt |
| `data/bracket_structure.json` | Estructura del cuadro de eliminatorias | Wikipedia (modelado a mano, verificado contra resultados reales) |
| `data/teams.csv` | Elo, ranking FIFA y grupo de las 48 selecciones | Kaggle (snapshot local, para que el notebook de modelo no dependa de `kagglehub`) |

**Dependencias:** `pip install kagglehub pandas beautifulsoup4 lxml requests nodriver`


## 1. Configuración

In [1]:
from pathlib import Path
import hashlib, re, time
import pandas as pd
import requests
from bs4 import BeautifulSoup

DATA_DIR = Path("data")
CACHE_DIR = Path("cache")
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "world_cup_matches.csv"
FORM_CSV_PATH = DATA_DIR / "pre_world_cup_form.csv"
BRACKET_JSON_PATH = DATA_DIR / "bracket_structure.json"
TEAMS_CSV_PATH = DATA_DIR / "teams.csv"

WC_START = "2026-06-11"       # los partidos "pre-Mundial" son anteriores a esta fecha
HTTP_DELAY = 3                # segundos mínimos entre peticiones web normales
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
}

# estadísticas por equipo en world_cup_matches.csv
STAT_NAMES = ["possession", "shots", "shots_on_target", "corners", "fouls",
              "offsides", "cards_yellow", "cards_red", "saves", "crosses",
              "interceptions", "tackles_won"]
STAT_COLS = [f"{side}_{s}" for side in ("home", "away") for s in STAT_NAMES]
FIXTURE_COLS = ["match_id", "date", "time_local", "round", "gameweek",
                "home_team", "away_team", "home_goals", "away_goals",
                "home_pens", "away_pens", "venue", "attendance", "referee",
                "notes", "match_url", "status", "stats_scraped", "stats_source",
                "home_xg", "away_xg", "competition", "competition_weight"]
INT_COLS = (["home_goals", "away_goals", "home_pens", "away_pens", "attendance"]
            + STAT_COLS)

# los nombres de equipo canónicos son los de Kaggle; el CSV heredado de FBref
# usa tres nombres distintos que normalizamos al cargar
FBREF_TO_CANON = {
    "Korea Republic": "South Korea",
    "Bosnia–Herz": "Bosnia and Herzegovina",
    "United States": "USA",
}


def cache_path(url):
    """Ruta de caché legible y única para una URL."""
    h = hashlib.md5(url.encode()).hexdigest()[:12]
    slug = re.sub(r"[^A-Za-z0-9]+", "_", re.sub(r"^https?://[^/]+", "", url)).strip("_")[:80]
    return CACHE_DIR / f"{slug}__{h}.html"


def http_get_cached(url, force=False):
    """GET con requests normal (sin bypass anti-bot), caché y pausa de cortesía.
    Se usa para Transfermarkt, que sí responde a clientes HTTP normales."""
    p = cache_path(url)
    if p.exists() and not force:
        return p.read_text(encoding="utf-8")
    r = requests.get(url, headers=HEADERS, timeout=30)
    time.sleep(HTTP_DELAY)
    if r.status_code != 200 or "Just a moment" in r.text[:5000]:
        raise RuntimeError(f"HTTP {r.status_code} / desafío anti-bot en {url}")
    p.write_text(r.text, encoding="utf-8")
    print(f"    descargado: {url}")
    return r.text

In [2]:
## clear_recent_cache — borra solo los ficheros recientes, no la caché histórica

def clear_recent_cache(days=3):
    """Elimina de cache/ los ficheros HTML modificados en los últimos `days` días.
    Útil para forzar la actualización de datos recientes sin perder la caché
    de partidos ya históricos (que son inmutables y caros de re-scrapear).
    """
    cutoff = time.time() - days * 86400
    deleted = []
    for p in CACHE_DIR.glob("*.html"):
        if p.stat().st_mtime > cutoff:
            p.unlink()
            deleted.append(p.name)
    print(f"clear_recent_cache({days}d): {len(deleted)} fichero(s) eliminado(s) de {CACHE_DIR}/")
    return deleted


# Ejemplo de uso (no se lanza en Run All):
# clear_recent_cache(days=3)  # borra solo los últimos 3 días de caché


## 2. Comprobación: ¿acepta FBref clientes HTTP normales?

FBref está detrás de Cloudflare y bloquea `requests` con un desafío JavaScript
(403 + "Just a moment..."). Lo comprobamos en cada ejecución porque es la razón
por la que el complemento de FBref (sección 6) necesita un navegador real
(`nodriver` + Edge) en vez de una petición normal.

In [3]:
def test_fbref_plain():
    url = "https://fbref.com/en/comps/1/schedule/World-Cup-Scores-and-Fixtures"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        challenge = "Just a moment" in r.text[:5000]
        print(f"FBref con requests normal: HTTP {r.status_code}"
              f"{' + desafío Cloudflare' if challenge else ''}")
        if r.status_code == 200 and not challenge:
            print("¡FBref vuelve a estar accesible sin bypass! El complemento podría simplificarse.")
        else:
            print("FBref sigue bloqueado para clientes HTTP normales -> el complemento necesita nodriver+Edge.")
    except Exception as e:
        print(f"FBref inaccesible: {type(e).__name__}: {e}")
    time.sleep(HTTP_DELAY)

test_fbref_plain()

FBref con requests normal: HTTP 403 + desafío Cloudflare
FBref sigue bloqueado para clientes HTTP normales -> el complemento necesita nodriver+Edge.


## 3. Dataset de Kaggle — `download_kaggle_dataset`

`kagglehub` descarga la **última versión** publicada (el dataset se actualiza a
diario; no requiere cuenta de Kaggle por ser público). Ficheros que usamos:

- `matches_detailed.csv` — fixture completo con nombres, estadio, marcador y **xG**
- `match_team_stats.csv` — posesión, tiros, tiros a puerta, córners, faltas,
  fueras de juego y paradas por equipo (fuentes: fifa.com / sofascore), y
  `last_updated` (usado en el diagnóstico de la sección 5)
- `match_events.csv` — eventos, de los que agregamos las **tarjetas** por equipo
- `teams.csv` — las 48 selecciones (Elo, ranking FIFA, grupo)

In [4]:
import kagglehub


def download_kaggle_dataset():
    """Descarga (o reutiliza) la última versión del dataset y carga sus CSV."""
    path = Path(kagglehub.dataset_download("mominullptr/fifa-world-cup-2026-dataset", force_download=True))
    print("Versión de Kaggle en:", path)
    k = {f.stem: pd.read_csv(f) for f in path.glob("*.csv")}
    print("Ficheros:", ", ".join(sorted(k)))
    return k


kaggle = download_kaggle_dataset()
print(f"\nPartidos: {len(kaggle['matches_detailed'])} "
      f"(completados: {(kaggle['matches_detailed'].status == 'Completed').sum()}) | "
      f"con stats de equipo: {kaggle['match_team_stats'].match_id.nunique()} | "
      f"selecciones: {len(kaggle['teams'])}")

100%|██████████| 362k/362k [00:00<00:00, 675kB/s]

Extracting files...
Versión de Kaggle en: C:\Users\MEDIA\.cache\kagglehub\datasets\mominullptr\fifa-world-cup-2026-dataset\versions\63
Ficheros: match_events, match_lineups, match_prediction_features, match_team_stats, matches, matches_detailed, player_stats, referees, squads_and_players, teams, tournament_stages, venues

Partidos: 104 (completados: 96) | con stats de equipo: 96 | selecciones: 48


## 4. Integración en `world_cup_matches.csv` — `update_from_kaggle`

Política de fusión (pensada para la ejecución diaria):

- El **fixture existente se conserva**; cada partido de Kaggle se casa con nuestra
  fila por pareja de equipos (probando también la orientación invertida) y fecha
  más cercana (±2 días).
- **Kaggle manda** en marcador, estado, **xG** y en sus 7 estadísticas
  (posesión, tiros, tiros a puerta, córners, faltas, fueras de juego, paradas).
  Las tarjetas se completan desde `match_events` solo si faltan.
- Lo que Kaggle no trae (centros, intercepciones, entradas, penaltis de tandas,
  asistencia, árbitro) se **conserva de FBref** si ya se había capturado antes
  (ver complemento opcional, sección 6).
- Partidos de Kaggle sin fila nuestra (rondas nuevas) se **añaden**.
- Las filas `scheduled` con más de 2 días de antigüedad que ninguna fuente haya
  confirmado se **eliminan** (cruces no confirmados que caducan solos).
- `stats_source` indica la procedencia: `kaggle`, `fbref` o `kaggle+fbref`.

In [5]:
KAGGLE_STAT_MAP = {  # columna en match_team_stats.csv -> nombre nuestro
    "possession_pct": "possession", "total_shots": "shots",
    "shots_on_target": "shots_on_target", "corners": "corners",
    "fouls": "fouls", "offsides": "offsides", "saves": "saves",
}


def _load_existing(csv_path):
    if not Path(csv_path).exists():
        cols = FIXTURE_COLS + STAT_COLS
        return pd.DataFrame(columns=cols)
    df = pd.read_csv(csv_path)
    for col in ("home_team", "away_team"):
        df[col] = df[col].replace(FBREF_TO_CANON)
    for col in ("stats_source", "home_xg", "away_xg", "competition", "competition_weight"):
        if col not in df.columns:  # columnas nuevas añadidas en versiones posteriores
            df[col] = None
    return df


def _kaggle_team_stats(k):
    """{(match_id, team_id): {stat: valor}} con stats + tarjetas de eventos."""
    out = {}
    for _, r in k["match_team_stats"].iterrows():
        d = {ours: r[kag] for kag, ours in KAGGLE_STAT_MAP.items() if pd.notna(r.get(kag))}
        out[(r["match_id"], r["team_id"])] = d
    ev = k["match_events"]
    cards = ev[ev["event_type"].isin(["Yellow Card", "Red Card"])]
    for (mid, tid), grp in cards.groupby(["match_id", "team_id"]):
        d = out.setdefault((mid, tid), {})
        d["cards_yellow"] = int((grp["event_type"] == "Yellow Card").sum())
        d["cards_red"] = int((grp["event_type"] == "Red Card").sum())
    return out


def _find_row(df, home, away, date):
    """Índice de la fila que casa con el partido (o None), y si está invertida."""
    for h, a, swapped in ((home, away, False), (away, home, True)):
        cand = df[(df["home_team"] == h) & (df["away_team"] == a)]
        if len(cand):
            diffs = (pd.to_datetime(cand["date"]) - pd.to_datetime(date)).dt.days.abs()
            if diffs.min() <= 2:
                return diffs.idxmin(), swapped
    return None, False


def update_from_kaggle(k, csv_path=CSV_PATH):
    df = _load_existing(csv_path)
    team_stats = _kaggle_team_stats(k)
    id2name = dict(zip(k["teams"]["team_id"], k["teams"]["team_name"]))
    name2id = {v: kk for kk, v in id2name.items()}
    added = updated = 0

    for _, m in k["matches_detailed"].iterrows():
        home, away = m["home_team_name"], m["away_team_name"]
        idx, swapped = _find_row(df, home, away, m["date"])
        if idx is None:
            df.loc[len(df)] = {c: None for c in df.columns}
            idx, swapped = df.index[-1], False
            df.loc[idx, ["date", "time_local", "round", "home_team", "away_team",
                         "venue", "referee", "notes"]] = [
                m["date"], m["kickoff_time_utc"], m["stage_name"], home, away,
                m["stadium_name"], m.get("referee_name"), None]
            added += 1
        else:
            updated += 1

        # lados en la orientación de NUESTRA fila
        sides = (("home", home), ("away", away)) if not swapped else (("home", away), ("away", home))
        played = pd.notna(m["home_score"])
        if played:
            scores = {home: m["home_score"], away: m["away_score"],
                      "xg_" + home: m["home_xg"], "xg_" + away: m["away_xg"]}
            for side, team in sides:
                df.loc[idx, f"{side}_goals"] = int(scores[team])
                if pd.notna(scores["xg_" + team]):
                    df.loc[idx, f"{side}_xg"] = float(scores["xg_" + team])
            df.loc[idx, "status"] = "played"
        elif pd.isna(df.loc[idx, "home_goals"]):
            df.loc[idx, "status"] = "scheduled"

        # estadísticas de equipo (Kaggle manda; tarjetas solo rellenan huecos)
        src_prev = df.loc[idx, "stats_source"]
        had_fbref = ("fbref" in str(src_prev)) if pd.notna(src_prev) \
            else pd.notna(df.loc[idx, "home_shots"])
        got_kaggle = False
        kag_mid = m["match_id"]
        for side, team in sides:
            s = team_stats.get((kag_mid, name2id.get(team)))
            if not s:
                continue
            got_kaggle = True
            for stat, val in s.items():
                col = f"{side}_{stat}"
                if stat in ("cards_yellow", "cards_red") and pd.notna(df.loc[idx, col]):
                    continue  # las tarjetas de FBref (oficiales) no se pisan
                df.loc[idx, col] = val
        if got_kaggle or had_fbref:
            df.loc[idx, "stats_source"] = ("kaggle+fbref" if got_kaggle and had_fbref
                                           else ("kaggle" if got_kaggle else "fbref"))

    # caducan las filas 'scheduled' viejas que ninguna fuente confirma
    stale = ((df["status"] == "scheduled")
             & (pd.to_datetime(df["date"]) < pd.Timestamp.now() - pd.Timedelta(days=2)))
    if stale.any():
        print(f"Eliminadas {stale.sum()} filas 'scheduled' caducadas (cruces no confirmados)")
        df = df[~stale].reset_index(drop=True)

    df["stats_scraped"] = df["home_shots"].notna() | df["home_possession"].notna()
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in ("home_xg", "away_xg"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = (df[FIXTURE_COLS + STAT_COLS]
          .sort_values(["date", "time_local"], na_position="last")
          .reset_index(drop=True))
    df.to_csv(csv_path, index=False)
    print(f"Casados con el fixture: {updated} | añadidos: {added}")
    print(f"Guardado {csv_path} ({len(df)} filas, {int(df['stats_scraped'].sum())} con stats, "
          f"{int(df['home_xg'].notna().sum())} con xG)")
    return df


wc = update_from_kaggle(kaggle)
wc.head()

Casados con el fixture: 98 | añadidos: 6
Guardado data\world_cup_matches.csv (112 filas, 96 con stats, 96 con xG)


,match_id,date,time_local,round,gameweek,home_team,away_team,home_goals,away_goals,home_pens,...,away_shots_on_target,away_corners,away_fouls,away_offsides,away_cards_yellow,away_cards_red,away_saves,away_crosses,away_interceptions,away_tackles_won
0,3c1e3816,2026-06-11,13:00,Group stage,1.0,Mexico,South Africa,2,0,<NA>,...,2,3,15,1,2,2,4,8,7,7
1,beebb792,2026-06-11,20:00,Group stage,1.0,South Korea,Czechia,2,1,<NA>,...,3,3,13,2,0,0,4,15,7,7
2,f6d2bd84,2026-06-12,15:00,Group stage,1.0,Canada,Bosnia and Herzegovina,1,1,<NA>,...,3,4,15,1,3,0,5,10,10,13
3,f6c0596c,2026-06-12,18:00,Group stage,1.0,USA,Paraguay,4,1,<NA>,...,1,1,17,3,5,0,9,5,9,16
4,58580af9,2026-06-13,12:00,Group stage,1.0,Qatar,Switzerland,1,1,<NA>,...,7,10,11,0,1,0,2,35,7,6


## 5. Diagnóstico: ¿cuánto tarda Kaggle en reflejar un partido jugado?

La descripción del dataset dice que se actualiza "a diario", pero no cuánto
tarda en aparecer un partido recién jugado. Lo medimos empíricamente: para los
últimos 10 partidos jugados, comparamos su fecha con `last_updated` de
`match_team_stats.csv` (el fichero de estadísticas, que es lo último en llegar).

In [6]:
def diagnose_kaggle_lag(k, n=10):
    """Para los últimos n partidos jugados, días entre la fecha del partido
    y el `last_updated` de sus estadísticas de equipo en Kaggle."""
    md_df = k["matches_detailed"]
    played = md_df[md_df.status == "Completed"].sort_values("date").tail(n).copy()
    last_upd = k["match_team_stats"].groupby("match_id")["last_updated"].max()
    played["stats_last_updated"] = played["match_id"].map(last_upd)
    played["dias_retraso"] = (pd.to_datetime(played["stats_last_updated"])
                              - pd.to_datetime(played["date"])).dt.days
    cols = ["match_id", "date", "home_team_name", "away_team_name",
           "stats_last_updated", "dias_retraso"]
    return played[cols].reset_index(drop=True)


kaggle_lag = diagnose_kaggle_lag(kaggle)
print(kaggle_lag.to_string(index=False))
print(f"\nRetraso medio: {kaggle_lag.dias_retraso.mean():.1f} días | "
      f"mediana: {kaggle_lag.dias_retraso.median():.1f} | "
      f"mínimo: {kaggle_lag.dias_retraso.min()} | máximo: {kaggle_lag.dias_retraso.max()}")

 match_id       date home_team_name away_team_name stats_last_updated  dias_retraso
       87 2026-07-04      Argentina     Cabo Verde         2026-07-04             0
       88 2026-07-04       Colombia          Ghana         2026-07-04             0
       89 2026-07-04       Paraguay         France         2026-07-04             0
       90 2026-07-04         Canada        Morocco         2026-07-04             0
       91 2026-07-05         Brazil         Norway         2026-07-06             1
       92 2026-07-06         Mexico        England         2026-07-06             0
       93 2026-07-06       Portugal          Spain         2026-07-06             0
       95 2026-07-07      Argentina          Egypt         2026-07-08             1
       94 2026-07-07            USA        Belgium         2026-07-07             0
       96 2026-07-07    Switzerland       Colombia         2026-07-08             1

Retraso medio: 0.3 días | mediana: 0.0 | mínimo: 0 | máximo: 1


## 6. Complemento opcional — FBref vía `nodriver` + Edge

Kaggle no trae centros, intercepciones, entradas ganadas, penaltis de tandas,
asistencia ni árbitro. `complement_with_fbref()` los rellena scrapeando FBref
con un navegador Edge real controlado por `nodriver` (la única forma de superar
el desafío de Cloudflare que confirma la sección 2) — el mismo scraper de la
Fase 1, ahora reutilizado como **fuente complementaria activa**, no descartada.

Solo completa huecos: nunca pisa lo que ya viene de Kaggle.

Requiere `pip install nodriver`.

In [7]:
import asyncio
import threading
import nodriver as uc

FBREF_BASE = "https://fbref.com"
FBREF_SCHEDULE_URL = FBREF_BASE + "/en/comps/1/schedule/World-Cup-Scores-and-Fixtures"
FBREF_DELAY = 6  # FBref banea a partir de ~10 peticiones/minuto
SCHEDULE_TTL = 6 * 3600       # caché del calendario: 6 horas
REPORT_EMPTY_TTL = 4 * 3600   # reintentar match reports sin stats a las 4 h
RECENT_MATCH_DAYS = 3          # partidos jugados hace ≤3 d -> report con TTL corto
_BROWSER_CANDIDATES = [
    r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
    r"C:\Program Files\Microsoft\Edge\Application\msedge.exe",
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
]
BROWSER_PATH = next((p for p in _BROWSER_CANDIDATES if Path(p).exists()), None)

_loop = None
_browser = None
_last_request = 0.0


def _get_loop():
    global _loop
    if _loop is None:
        _loop = asyncio.ProactorEventLoop()  # en Windows, necesario para subprocesos
        threading.Thread(target=_loop.run_forever, daemon=True).start()
    return _loop


def _run(coro, timeout=240):
    return asyncio.run_coroutine_threadsafe(coro, _get_loop()).result(timeout)


async def _get_browser():
    global _browser
    if _browser is None:
        _browser = await uc.start(browser_executable_path=BROWSER_PATH, headless=False)
    return _browser


def close_browser():
    global _browser
    if _browser is not None:
        res = _browser.stop()
        if asyncio.iscoroutine(res):
            _run(res)
        _browser = None


async def _fetch(url):
    global _last_request
    browser = await _get_browser()
    wait = FBREF_DELAY - (time.monotonic() - _last_request)
    if wait > 0:
        await asyncio.sleep(wait)
    page = await browser.get(url)
    html = ""
    for _ in range(30):  # hasta ~90 s por si el desafío de Cloudflare tarda
        await asyncio.sleep(3)
        html = await page.get_content()
        if "Just a moment" not in html[:3000] and len(html) > 30000:
            break
    _last_request = time.monotonic()
    if "Just a moment" in html[:3000]:
        raise RuntimeError(f"Cloudflare no resuelto para {url}")
    return html


def _cache_stale(p, ttl_seconds):
    """True si el fichero de caché es más antiguo que ttl_seconds."""
    return not p.exists() or (time.time() - p.stat().st_mtime) > ttl_seconds


def fetch_with_cache(url, force=False, ttl=None):
    """GET con caché. ttl=None -> nunca caduca; ttl=N -> recarga si >N segundos."""
    p = cache_path(url)
    expired = ttl is not None and _cache_stale(p, ttl)
    if p.exists() and not force and not expired:
        return p.read_text(encoding="utf-8")
    html = _run(_fetch(url))
    p.write_text(html, encoding="utf-8")
    return html


In [8]:
# --- parseo del calendario y los match reports de FBref ---
SCORE_RE = re.compile(r"(?:\((\d+)\))?\s*(\d+)\s*[–\-]\s*(\d+)\s*(?:\((\d+)\))?")
SUMMARY_STATS = ["shots", "shots_on_target", "fouls", "offsides", "crosses",
                 "cards_yellow", "cards_red", "tackles_won", "interceptions"]
FBREF_ONLY_COLS = ["attendance", "referee", "notes", "home_pens", "away_pens",
                   "home_crosses", "away_crosses", "home_interceptions", "away_interceptions",
                   "home_tackles_won", "away_tackles_won",
                   "home_cards_yellow", "away_cards_yellow", "home_cards_red", "away_cards_red"]


def _clean_team(cell):
    a = cell.find("a")
    return (a or cell).get_text(strip=True)


def parse_schedule(html):
    """Tabla Scores & Fixtures de FBref -> DataFrame (una fila por partido)."""
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table", id="sched_all") or soup.find(
        "table", id=lambda x: x and x.startswith("sched"))
    if table is None:
        raise ValueError("No se encontró la tabla de calendario")
    rows = []
    for tr in table.find("tbody").find_all("tr"):
        if tr.get("class") and "thead" in tr["class"]:
            continue
        cells = {c.get("data-stat"): c for c in tr.find_all(["th", "td"])}
        home_c, away_c = cells.get("home_team"), cells.get("away_team")
        if home_c is None or not home_c.get_text(strip=True):
            continue
        m = SCORE_RE.search(cells["score"].get_text(strip=True)) if cells.get("score") else None
        hp, hg, ag, ap = m.groups() if m else (None, None, None, None)
        match_url = None
        mr = cells.get("match_report")
        if mr and mr.get_text(strip=True) == "Match Report" and mr.find("a"):
            match_url = FBREF_BASE + mr.find("a")["href"]
        att = cells["attendance"].get_text(strip=True).replace(",", "") if cells.get("attendance") else ""
        time_txt = cells["start_time"].get_text(strip=True) if cells.get("start_time") else ""
        rows.append({
            "match_id": match_url.split("/matches/")[1].split("/")[0] if match_url else None,
            "date": cells["date"].get_text(strip=True) if cells.get("date") else "",
            "time_local": time_txt.split("(")[0],
            "round": cells["round"].get_text(strip=True) if cells.get("round") else "",
            "gameweek": cells["gameweek"].get_text(strip=True) if cells.get("gameweek") else "",
            "home_team": _clean_team(home_c),
            "away_team": _clean_team(away_c),
            "home_goals": int(hg) if hg is not None else None,
            "away_goals": int(ag) if ag is not None else None,
            "home_pens": int(hp) if hp is not None else None,
            "away_pens": int(ap) if ap is not None else None,
            "venue": cells["venue"].get_text(strip=True) if cells.get("venue") else "",
            "attendance": int(att) if att.isdigit() else None,
            "referee": cells["referee"].get_text(strip=True) if cells.get("referee") else "",
            "notes": cells["notes"].get_text(strip=True) if cells.get("notes") else "",
            "match_url": match_url,
            "status": "played" if m else "scheduled",
        })
    return pd.DataFrame(rows)


def _summary_totals(table):
    out = {}
    tf = table.find("tfoot")
    if not tf:
        return out
    for c in tf.find_all(["th", "td"]):
        ds = c.get("data-stat")
        if ds in SUMMARY_STATS:
            txt = c.get_text(strip=True)
            out[ds] = int(txt) if txt.lstrip("-").isdigit() else None
    return out


def _possession(soup):
    div = soup.find("div", id="team_stats")
    if not div:
        return None, None
    for tr in div.find_all("tr"):
        if tr.get_text(strip=True) == "Possession":
            vals = re.findall(r"(\d+)%", tr.find_next_sibling("tr").get_text())
            if len(vals) >= 2:
                return int(vals[0]), int(vals[1])
    return None, None


def _team_stats_extra(soup):
    out = {}
    div = soup.find("div", id="team_stats_extra")
    if not div:
        return out
    for block in div.find_all("div", recursive=False):
        vals = [d.get_text(strip=True) for d in block.find_all("div", recursive=False)]
        trip = vals[3:]  # los tres primeros divs son la cabecera
        for i in range(0, len(trip) - 2, 3):
            h, name, a = trip[i], trip[i + 1], trip[i + 2]
            if name and h.isdigit() and a.isdigit():
                out[name.lower()] = (int(h), int(a))
    return out


def _saves(soup, team_id):
    t = soup.find("table", id=f"keeper_stats_{team_id}")
    if not t or not t.find("tbody"):
        return None
    vals = [c.get_text(strip=True) for c in t.find("tbody").find_all("td", {"data-stat": "gk_saves"})]
    vals = [int(v) for v in vals if v.isdigit()]
    return sum(vals) if vals else None


def parse_match_report(html):
    """Match report de FBref -> dict {home_*/away_*} de stats de equipo."""
    soup = BeautifulSoup(html, "lxml")
    stats = {}
    tables = soup.find_all("table", id=re.compile(r"^stats_[0-9a-f]+_summary$"))
    team_ids = []
    for side, table in zip(("home", "away"), tables[:2]):
        team_ids.append(table["id"].split("_")[1])
        for kk, v in _summary_totals(table).items():
            stats[f"{side}_{kk}"] = v
    stats["home_possession"], stats["away_possession"] = _possession(soup)
    extra = _team_stats_extra(soup)
    for name in ("corners", "fouls", "offsides", "crosses", "interceptions"):
        if name in extra:
            h, a = extra[name]
            if stats.get(f"home_{name}") is None:
                stats[f"home_{name}"] = h
            if stats.get(f"away_{name}") is None:
                stats[f"away_{name}"] = a
    for side, tid in zip(("home", "away"), team_ids):
        stats[f"{side}_saves"] = _saves(soup, tid)
    return stats

In [9]:
def complement_with_fbref(csv_path=CSV_PATH, refresh_schedule=True):
    """Complementa world_cup_matches.csv con datos de FBref.

    Política (Problem 3 fix):
    - Si FBref tiene marcador para una fila 'scheduled', actualiza score+status.
    - Si FBref tiene un partido que el CSV no tiene (Kaggle aún no lo publicó),
      lo añade con stats_source='fbref' para que aparezca de inmediato.
    - Solo rellena huecos para columnas que Kaggle no trae (centros, tarjetas,
      intercepciones, entradas, asistencia, árbitro, penaltis de tandas).
    - Nunca pisa marcadores ya presentes desde Kaggle.
    """
    df = pd.read_csv(csv_path)
    sched_html = fetch_with_cache(FBREF_SCHEDULE_URL,
                                   force=refresh_schedule,
                                   ttl=SCHEDULE_TTL)
    sched = parse_schedule(sched_html)
    filled = updated_scores = added_from_fbref = 0

    for _, s in sched.iterrows():
        home_c = FBREF_TO_CANON.get(s.home_team, s.home_team)
        away_c = FBREF_TO_CANON.get(s.away_team, s.away_team)
        idx, swapped = _find_row(df, home_c, away_c, s.date)

        if idx is None:
            # FBref conoce este partido pero Kaggle todavía no -> añadirlo
            if s.status != "played" or pd.isna(s.home_goals):
                continue
            df.loc[len(df)] = {c: None for c in df.columns}
            idx = df.index[-1]
            swapped = False
            df.loc[idx, ["date", "time_local", "round", "home_team", "away_team",
                          "venue", "referee", "notes", "attendance", "match_url"]] = [
                s.date, s.time_local, s.round, home_c, away_c,
                s.venue, getattr(s, "referee", None),
                getattr(s, "notes", None), getattr(s, "attendance", None), s.match_url]
            df.loc[idx, "home_goals"] = int(s.home_goals)
            df.loc[idx, "away_goals"] = int(s.away_goals)
            if pd.notna(getattr(s, "home_pens", None)):
                df.loc[idx, "home_pens"] = int(s.home_pens)
                df.loc[idx, "away_pens"] = int(s.away_pens)
            df.loc[idx, "status"] = "played"
            df.loc[idx, "stats_source"] = "fbref"
            added_from_fbref += 1

        sides = (("home", "home"), ("away", "away")) if not swapped                 else (("home", "away"), ("away", "home"))

        # Actualizar score si CSV aún tiene 'scheduled' pero FBref ya tiene resultado
        if (s.status == "played" and pd.notna(getattr(s, "home_goals", None))
                and df.loc[idx, "status"] != "played"):
            h_g = int(s.home_goals) if not swapped else int(s.away_goals)
            a_g = int(s.away_goals) if not swapped else int(s.home_goals)
            df.loc[idx, ["home_goals", "away_goals", "status"]] = [h_g, a_g, "played"]
            h_p = getattr(s, "home_pens", None)
            a_p = getattr(s, "away_pens", None)
            if pd.notna(h_p) and pd.notna(a_p):
                df.loc[idx, "home_pens"] = int(h_p) if not swapped else int(a_p)
                df.loc[idx, "away_pens"] = int(a_p) if not swapped else int(h_p)
            src_prev = str(df.loc[idx, "stats_source"]) if pd.notna(df.loc[idx, "stats_source"]) else ""
            if "fbref" not in src_prev:
                df.loc[idx, "stats_source"] = (src_prev + "+fbref").lstrip("+") if src_prev else "fbref"
            updated_scores += 1

        if s.status != "played":
            continue

        # Huecos: asistencia, árbitro, notas, penaltis de tandas
        for our_col, fb_col in [("attendance", "attendance"),
                                  ("referee", "referee"), ("notes", "notes")]:
            if pd.isna(df.loc[idx, our_col]) and hasattr(s, fb_col):
                val = getattr(s, fb_col)
                if pd.notna(val):
                    df.loc[idx, our_col] = val
        for our_side, fb_side in sides:
            p_col = f"{our_side}_pens"
            if pd.isna(df.loc[idx, p_col]):
                val = getattr(s, f"{fb_side}_pens", None)
                if pd.notna(val):
                    df.loc[idx, p_col] = val

        # Match report para centros, intercepciones, entradas, tarjetas
        needs = any(pd.isna(df.loc[idx, f"{side}_{stat}"])
                    for side in ("home", "away")
                    for stat in ("crosses", "interceptions", "tackles_won",
                                 "cards_yellow", "cards_red"))
        if not needs or not s.match_url:
            continue

        days_since = (pd.Timestamp.now() - pd.to_datetime(s.date)).days
        report_ttl = REPORT_EMPTY_TTL if days_since <= RECENT_MATCH_DAYS else None
        report_html = fetch_with_cache(s.match_url, ttl=report_ttl)
        report_stats = parse_match_report(report_html)
        if not report_stats:
            continue
        for our_side, fb_side in sides:
            for stat in ("crosses", "interceptions", "tackles_won",
                         "cards_yellow", "cards_red"):
                col = f"{our_side}_{stat}"
                val = report_stats.get(f"{fb_side}_{stat}")
                if val is not None and pd.isna(df.loc[idx, col]):
                    df.loc[idx, col] = val
        filled += 1

    # Actualiza stats_source donde proceda
    src_col = df["stats_source"].fillna("")
    mask = src_col.str.len() > 0
    df.loc[mask, "stats_source"] = src_col[mask].apply(
        lambda v: v if "fbref" in v else v + "+fbref")

    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in ("home_xg", "away_xg"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["stats_scraped"] = df["home_shots"].notna() | df["home_possession"].notna()
    df = (df[FIXTURE_COLS + STAT_COLS]
          .sort_values(["date", "time_local"], na_position="last")
          .reset_index(drop=True))
    df.to_csv(csv_path, index=False)
    print(f"complement_with_fbref: match reports completados en {filled} partido(s)")
    if updated_scores:
        print(f"  Marcadores actualizados desde FBref (Kaggle no los tenía aun): {updated_scores}")
    if added_from_fbref:
        print(f"  Partidos añadidos desde FBref (ausentes en Kaggle): {added_from_fbref}")
    return df


> ⚠️ **EJECUCIÓN MANUAL — no se lanza con "Run All".** Abre una ventana de Edge
> y tarda ~6 s por partido nuevo (Cloudflare). Descomenta la línea siguiente y
> ejecuta la celda a mano cuando quieras completar esas columnas.

In [10]:
wc = complement_with_fbref()  # descomenta para ejecutar manualmente
close_browser()

complement_with_fbref: match reports completados en 2 partido(s)


## 7. Forma pre-Mundial: Transfermarkt

Los **últimos 5 partidos antes del Mundial** de cada selección salen de
**Transfermarkt**, que responde a `requests` con User-Agent de navegador (su
`robots.txt` lo permite; pausa de ≥3 s por petición).

Dos pasos:
1. **IDs de equipo**: del ranking FIFA de TM (6 páginas × 25 selecciones),
   casando nombres con alias (`Türkiye→Turkiye`, `Cabo Verde→Cape Verde`...).
2. **Partidos**: la página `spielplandatum` de cada selección (temporada 2025/26,
   con 2024/25 de respaldo si no llegan a 5 partidos pre-Mundial).

Limitación: TM no publica estadísticas de equipo (posesión, tiros…) para
partidos de selecciones, así que este CSV lleva fixture y resultado, no stats.

In [11]:
TM_BASE = "https://www.transfermarkt.com"
TM_ALIASES = {  # nombre Kaggle -> nombre en Transfermarkt
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    "Congo DR": "Democratic Republic of the Congo",
    "Côte d'Ivoire": "Ivory Coast",
    "IR Iran": "Iran",
    "Türkiye": "Turkiye",
    "USA": "United States",
}


def get_tm_team_map(wc_teams):
    """{selección (nombre Kaggle): (slug, id de Transfermarkt)}"""
    tm = {}
    for page in range(1, 7):
        html = http_get_cached(f"{TM_BASE}/statistik/weltrangliste?page={page}")
        for a in BeautifulSoup(html, "lxml").find_all(
                "a", href=re.compile(r"/startseite/verein/\d+")):
            mm = re.search(r"/([^/]+)/startseite/verein/(\d+)", a["href"])
            name = a.get("title") or a.get_text(strip=True)
            if mm and name:
                tm[name] = (mm.group(1), mm.group(2))
    mapping = {}
    for team in wc_teams:
        tm_name = TM_ALIASES.get(team, team)
        if tm_name in tm:
            mapping[team] = tm[tm_name]
    missing = sorted(set(wc_teams) - set(mapping))
    print(f"Selecciones mapeadas a Transfermarkt: {len(mapping)}/{len(wc_teams)}"
          + (f" | SIN MAPEAR: {missing}" if missing else ""))
    return mapping


WC_TEAMS = sorted(kaggle["teams"]["team_name"])
tm_map = get_tm_team_map(WC_TEAMS)

Selecciones mapeadas a Transfermarkt: 48/48


In [12]:
TM_DATE_RE = re.compile(r"(\d{2})/(\d{2})/(\d{2})")
TM_SCORE_RE = re.compile(r"(\d+):(\d+)")
TM_FORM_COLS = ["team", "match_date", "opponent", "venue", "competition",
                "goals_for", "goals_against", "result", "extra_time", "penalties"]


def parse_tm_fixtures(html, team_name):
    """Página spielplandatum de TM -> DataFrame de partidos jugados del equipo."""
    soup = BeautifulSoup(html, "lxml")
    matches = []
    for table in soup.find_all("table"):
        head = table.find("tr")
        if not head or "Matchday" not in head.get_text():
            continue
        comp = ""
        for tr in table.find_all("tr")[1:]:
            tds = tr.find_all("td")
            texts = [td.get_text(" ", strip=True) for td in tds]
            if len(tds) <= 2:  # fila-cabecera con el nombre de la competición
                comp = tr.get_text(" ", strip=True) or comp
                continue
            dm = TM_DATE_RE.search(" | ".join(texts))
            if not dm:
                continue
            d, mth, y = dm.groups()
            venue = next((t for t in texts if t in ("H", "A", "N")), None)
            opp_a = next((td.find("a", href=re.compile(r"/verein/\d+"))
                          for td in tds if td.find("a", href=re.compile(r"/verein/\d+"))), None)
            # marcador SOLO de la última celda (columna Result): la hora de
            # inicio ('7:00 PM') también casa con la regex de marcador
            result_txt = texts[-1].strip()
            sm = TM_SCORE_RE.match(result_txt)
            if not (venue and opp_a is not None and sm):
                continue  # partido sin jugar ('-:-'), aplazado o fila rara
            opponent = re.sub(r"\s*\(\d+\.\)$", "",
                              opp_a.get("title") or opp_a.get_text(strip=True)).strip()
            home_g, away_g = int(sm.group(1)), int(sm.group(2))
            gf, ga = (home_g, away_g) if venue == "H" else (away_g, home_g)
            matches.append({
                "team": team_name,
                "match_date": f"20{y}-{mth}-{d}",
                "opponent": opponent,
                "venue": {"H": "home", "A": "away", "N": "neutral"}[venue],
                "competition": comp,
                "goals_for": gf, "goals_against": ga,
                "result": "W" if gf > ga else ("L" if gf < ga else "D"),
                "extra_time": "aet" in result_txt.lower(),
                "penalties": "pen" in result_txt.lower(),
            })
    return pd.DataFrame(matches, columns=TM_FORM_COLS)


def last5_before_wc(team, slug, tm_id):
    """Últimos 5 partidos del equipo anteriores al Mundial (temporadas 25/26 y 24/25)."""
    parts = []
    for saison in (2025, 2024):
        url = f"{TM_BASE}/{slug}/spielplandatum/verein/{tm_id}/saison_id/{saison}"
        parts.append(parse_tm_fixtures(http_get_cached(url), team))
        pre = (pd.concat(parts, ignore_index=True)
               .query("match_date < @WC_START")
               .drop_duplicates(subset=["match_date", "opponent"])
               .sort_values("match_date"))
        if len(pre) >= 5:
            break
    return pre.tail(5)

## 8. Descarga de los últimos 5 partidos de las 48 selecciones

In [13]:
form_parts, incomplete = [], {}
for i, team in enumerate(WC_TEAMS, 1):
    if team not in tm_map:
        incomplete[team] = 0
        continue
    slug, tm_id = tm_map[team]
    last5 = last5_before_wc(team, slug, tm_id)
    print(f"[{i:2}/48] {team}: {len(last5)} partidos pre-Mundial", flush=True)
    if len(last5) < 5:
        incomplete[team] = len(last5)
    form_parts.append(last5)

form = pd.concat(form_parts, ignore_index=True)
form["source"] = "transfermarkt.com"
form.to_csv(FORM_CSV_PATH, index=False)
print(f"\nGuardado {FORM_CSV_PATH} ({len(form)} filas)")
print("Selecciones con los 5 completos:", 48 - len(incomplete),
      "| incompletas:", incomplete or "ninguna")

[ 1/48] Algeria: 5 partidos pre-Mundial
[ 2/48] Argentina: 5 partidos pre-Mundial
[ 3/48] Australia: 5 partidos pre-Mundial
[ 4/48] Austria: 5 partidos pre-Mundial
[ 5/48] Belgium: 5 partidos pre-Mundial
[ 6/48] Bosnia and Herzegovina: 5 partidos pre-Mundial
[ 7/48] Brazil: 5 partidos pre-Mundial
[ 8/48] Cabo Verde: 5 partidos pre-Mundial
[ 9/48] Canada: 5 partidos pre-Mundial
[10/48] Colombia: 5 partidos pre-Mundial
[11/48] Congo DR: 5 partidos pre-Mundial
[12/48] Croatia: 5 partidos pre-Mundial
[13/48] Curaçao: 5 partidos pre-Mundial
[14/48] Czechia: 5 partidos pre-Mundial
[15/48] Côte d'Ivoire: 5 partidos pre-Mundial
[16/48] Ecuador: 5 partidos pre-Mundial
[17/48] Egypt: 5 partidos pre-Mundial
[18/48] England: 5 partidos pre-Mundial
[19/48] France: 5 partidos pre-Mundial
[20/48] Germany: 5 partidos pre-Mundial
[21/48] Ghana: 5 partidos pre-Mundial
[22/48] Haiti: 5 partidos pre-Mundial
[23/48] IR Iran: 5 partidos pre-Mundial
[24/48] Iraq: 5 partidos pre-Mundial
[25/48] Japan: 5 parti

## 9. Estructura del cuadro de eliminatorias — `data/bracket_structure.json`

Fuente: [Wikipedia — 2026 FIFA World Cup knockout stage](https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_knockout_stage)
(la numeración oficial de partidos 73-104 y el reparto de terceros surge del
Anexo C del reglamento de la FIFA, resumido en esa página). Los grupos y el
nombre de cada selección salen de `teams.csv` (Kaggle).

**Verificación**: sus 16 emparejamientos de dieciseisavos (partidos 73-88)
coinciden 16/16 contra los partidos ya jugados de esa ronda en `world_cup_matches.csv`.

**Regla de los mejores terceros**: los 2 primeros de cada uno de los 12 grupos
clasifican directos; de los 12 terceros, los **8 mejor clasificados** (por
puntos, luego diferencia de goles, luego goles a favor) completan el cuadro.

**Tabla de combinaciones (Anexo C)**: la FIFA publica 495 combinaciones posibles;
como la fase de grupos de este Mundial ya está completa e inmutable, solo
modelamos la combinación que realmente se dio (terceros de B, D, E, F, I, J, K, L).

In [14]:
import json

GROUPS = {g: grp.sort_values("team_id").team_name.tolist()
          for g, grp in kaggle["teams"].groupby("group_letter")}

# Partidos 73-88: combinación real de terceros de este Mundial (B,D,E,F,I,J,K,L)
ROUND_OF_32 = [
    {"match_number": 73, "home_slot": "2A", "away_slot": "2B"},
    {"match_number": 74, "home_slot": "1E", "away_slot": "3D"},
    {"match_number": 75, "home_slot": "1F", "away_slot": "2C"},
    {"match_number": 76, "home_slot": "1C", "away_slot": "2F"},
    {"match_number": 77, "home_slot": "1I", "away_slot": "3F"},
    {"match_number": 78, "home_slot": "2E", "away_slot": "2I"},
    {"match_number": 79, "home_slot": "1A", "away_slot": "3E"},
    {"match_number": 80, "home_slot": "1L", "away_slot": "3K"},
    {"match_number": 81, "home_slot": "1D", "away_slot": "3B"},
    {"match_number": 82, "home_slot": "1G", "away_slot": "3I"},
    {"match_number": 83, "home_slot": "2K", "away_slot": "2L"},
    {"match_number": 84, "home_slot": "1H", "away_slot": "2J"},
    {"match_number": 85, "home_slot": "1B", "away_slot": "3J"},
    {"match_number": 86, "home_slot": "1J", "away_slot": "2H"},
    {"match_number": 87, "home_slot": "1K", "away_slot": "3L"},
    {"match_number": 88, "home_slot": "2D", "away_slot": "2G"},
]

# Progresión: partido 89 en adelante, siempre "ganador/perdedor del partido N"
PROGRESSION = [
    # Round of 16 - pares verificados contra octavos reales (FIFA, 3-jul-2026)
    # NO son consecutivos: cuadro cruza 73+75, 74+77, 76+78, 85+87, 86+88
    {"match_number": 89, "round": "Round of 16", "home_source": "W73", "away_source": "W75"},
    {"match_number": 90, "round": "Round of 16", "home_source": "W74", "away_source": "W77"},
    {"match_number": 91, "round": "Round of 16", "home_source": "W76", "away_source": "W78"},
    {"match_number": 92, "round": "Round of 16", "home_source": "W79", "away_source": "W80"},
    {"match_number": 93, "round": "Round of 16", "home_source": "W81", "away_source": "W82"},
    {"match_number": 94, "round": "Round of 16", "home_source": "W83", "away_source": "W84"},
    {"match_number": 95, "round": "Round of 16", "home_source": "W85", "away_source": "W87"},
    {"match_number": 96, "round": "Round of 16", "home_source": "W86", "away_source": "W88"},
    {"match_number": 97,  "round": "Quarterfinal", "home_source": "W89", "away_source": "W90"},
    {"match_number": 98,  "round": "Quarterfinal", "home_source": "W91", "away_source": "W92"},
    {"match_number": 99,  "round": "Quarterfinal", "home_source": "W93", "away_source": "W94"},
    {"match_number": 100, "round": "Quarterfinal", "home_source": "W95", "away_source": "W96"},
    {"match_number": 101, "round": "Semifinal",    "home_source": "W97", "away_source": "W98"},
    {"match_number": 102, "round": "Semifinal",    "home_source": "W99", "away_source": "W100"},
    {"match_number": 103, "round": "Third place",  "home_source": "L101", "away_source": "L102"},
    {"match_number": 104, "round": "Final",        "home_source": "W101", "away_source": "W102"},
]

bracket_structure = {
    "source": "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_knockout_stage",
    "verified_against": "world_cup_matches.csv (16/16 partidos de dieciseisavos ya jugados coinciden)",
    "groups": GROUPS,
    "third_place_ranking_criteria": ["points", "goal_difference", "goals_for"],
    "third_place_ranking_note": ("Simplificación de los desempates completos de la FIFA "
                                 "(que también usan enfrentamiento directo, fair play y sorteo); "
                                 "estos 3 criterios ya reproducen el resultado real de este Mundial."),
    "qualified_third_place_groups_2026": ["B", "D", "E", "F", "I", "J", "K", "L"],
    "third_place_combination_note": ("El Anexo C del reglamento FIFA define 495 combinaciones "
                                     "posibles; aquí solo se modela la que realmente se dio "
                                     "(fase de grupos ya completa e inmutable)."),
    "round_of_32": ROUND_OF_32,
    "progression": PROGRESSION,
}
json.dump(bracket_structure, open(BRACKET_JSON_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"Guardado {BRACKET_JSON_PATH}")
for g, teams_ in GROUPS.items():
    print(f"  Grupo {g}: {', '.join(teams_)}")

Guardado data\bracket_structure.json
  Grupo A: Mexico, South Africa, South Korea, Czechia
  Grupo B: Canada, Bosnia and Herzegovina, Qatar, Switzerland
  Grupo C: Brazil, Morocco, Haiti, Scotland
  Grupo D: USA, Paraguay, Australia, Türkiye
  Grupo E: Germany, Curaçao, Côte d'Ivoire, Ecuador
  Grupo F: Netherlands, Japan, Sweden, Tunisia
  Grupo G: Belgium, Egypt, IR Iran, New Zealand
  Grupo H: Spain, Cabo Verde, Saudi Arabia, Uruguay
  Grupo I: France, Senegal, Iraq, Norway
  Grupo J: Argentina, Algeria, Austria, Jordan
  Grupo K: Portugal, Congo DR, Uzbekistan, Colombia
  Grupo L: England, Croatia, Ghana, Panama


### `compute_group_standings`, ranking de terceros y `get_bracket_state()`

Igual que en la fase anterior del proyecto: resuelve cada casilla del cuadro
(grupos -> dieciseisavos -> ... -> final) cruzando `bracket_structure.json`
con `world_cup_matches.csv`, sin inventar nunca un cruce que el CSV no confirme.
Estas mismas funciones se duplican en `mundial2026_modelo.ipynb` (son puro
cálculo local, sin red) para que ese notebook pueda localizar cruces pendientes
sin depender de este.

In [15]:
def compute_group_standings(wc_df):
    """Clasificación de cada grupo a partir de los partidos de grupo ya jugados."""
    gs = wc_df[(wc_df["round"] == "Group stage") & (wc_df["status"] == "played")]
    rows = []
    for _, r in gs.iterrows():
        rows.append((r.home_team, r.home_goals, r.away_goals))
        rows.append((r.away_team, r.away_goals, r.home_goals))
    tbl = pd.DataFrame(rows, columns=["team", "gf", "ga"])
    tbl["pts"] = tbl.apply(lambda x: 3 if x.gf > x.ga else (1 if x.gf == x.ga else 0), axis=1)
    agg = tbl.groupby("team").agg(gf=("gf", "sum"), ga=("ga", "sum"),
                                  played=("gf", "size"), pts=("pts", "sum")).reset_index()
    agg["gd"] = agg.gf - agg.ga
    team2group = {t: g for g, teams_ in GROUPS.items() for t in teams_}
    agg["group"] = agg["team"].map(team2group)
    agg = agg.sort_values(["group", "pts", "gd", "gf"], ascending=[True, False, False, False])
    agg["position"] = agg.groupby("group").cumcount() + 1
    agg["group_complete"] = agg.groupby("group")["played"].transform(lambda s: (s == 3).all())
    return agg.reset_index(drop=True)


def rank_third_place(standings):
    """Los 12 terceros de grupo, ordenados para saber cuáles 8 pasan."""
    thirds = standings[(standings.position == 3) & standings.group_complete].copy()
    thirds = thirds.sort_values(["pts", "gd", "gf"], ascending=[False, False, False]).reset_index(drop=True)
    thirds["third_rank"] = thirds.index + 1
    return thirds


def resolve_group_slot(slot, standings):
    pos, group = int(slot[0]), slot[1]
    g = standings[standings.group == group]
    if g.empty or not g["group_complete"].iloc[0]:
        return None
    row = g[g.position == pos]
    return row.iloc[0].team if len(row) else None


def resolve_third_slot(group, standings, qualified_thirds):
    if group not in qualified_thirds:
        return None
    g = standings[(standings.group == group) & (standings.position == 3)]
    if g.empty or not g["group_complete"].iloc[0]:
        return None
    return g.iloc[0].team


def _resolve_slot(slot, standings, qualified_thirds):
    return (resolve_group_slot(slot, standings) if slot[0] in "12"
            else resolve_third_slot(slot[1], standings, qualified_thirds))


def _resolve_source(src, rows_by_num):
    kind, num = src[0], int(src[1:])
    feeder = rows_by_num.get(num)
    if not feeder or not feeder["winner"]:
        return None
    return feeder["winner"] if kind == "W" else feeder["loser"]


def _describe_source(src, rows_by_num):
    kind, num = src[0], int(src[1:])
    label = "Ganador" if kind == "W" else "Perdedor"
    feeder = rows_by_num.get(num)
    if feeder and feeder["confirmed_teams"]:
        return f"{label} de Partido {num} ({feeder['home_team']} vs {feeder['away_team']})"
    return f"{label} de Partido {num}"


def _find_match_row(wc_df, round_name, home, away):
    cand = wc_df[(wc_df["round"] == round_name) &
                (((wc_df.home_team == home) & (wc_df.away_team == away)) |
                 ((wc_df.home_team == away) & (wc_df.away_team == home)))]
    return cand.iloc[0] if len(cand) else None


def _find_conflicts(wc_df, round_name, home, away):
    involved = wc_df[(wc_df["round"] == round_name) &
                     (wc_df.home_team.isin([home, away]) | wc_df.away_team.isin([home, away]))]
    return [f"{r.home_team} vs {r.away_team} ({r.status})" for _, r in involved.iterrows()
            if {r.home_team, r.away_team} != {home, away}]


def _settle_match(wc_df, match_number, round_name, home_slot, away_slot, home, away, rows_by_num):
    row = {"match_number": match_number, "round": round_name,
           "home_slot": home_slot, "away_slot": away_slot,
           "winner": None, "loser": None}

    if home is None or away is None:
        row["home_team"] = home or _describe_source(home_slot, rows_by_num)
        row["away_team"] = away or _describe_source(away_slot, rows_by_num)
        row["confirmed_teams"] = False
        row["status"] = "condicional"
        row["result"] = None
        row["confirmed_by_data"] = False
        row["discrepancy"] = None
        rows_by_num[match_number] = row
        return row

    row["home_team"], row["away_team"], row["confirmed_teams"] = home, away, True
    match_row = _find_match_row(wc_df, round_name, home, away)
    conflicts = _find_conflicts(wc_df, round_name, home, away)
    row["discrepancy"] = ("El CSV tiene en esta ronda otra fila con uno de estos equipos pero "
                          f"rival distinto: {'; '.join(conflicts)}") if conflicts else None

    if match_row is None:
        row["status"], row["result"], row["confirmed_by_data"] = "pendiente (sin fila en el CSV)", None, False
    elif match_row["status"] != "played":
        row["status"], row["result"], row["confirmed_by_data"] = "scheduled", None, True
    else:
        row["confirmed_by_data"] = True
        if match_row.home_team == home:
            h_g, a_g = match_row.home_goals, match_row.away_goals
            h_p, a_p = match_row.get("home_pens"), match_row.get("away_pens")
        else:
            h_g, a_g = match_row.away_goals, match_row.home_goals
            h_p, a_p = match_row.get("away_pens"), match_row.get("home_pens")
        if h_g > a_g:
            row["winner"], row["loser"] = home, away
        elif a_g > h_g:
            row["winner"], row["loser"] = away, home
        elif pd.notna(h_p) and pd.notna(a_p):
            row["winner"], row["loser"] = (home, away) if h_p > a_p else (away, home)
        row["status"] = "played"
        row["result"] = f"{h_g:.0f}-{a_g:.0f}" + (f" (pen. {h_p:.0f}-{a_p:.0f})" if pd.notna(h_p) else "")
    rows_by_num[match_number] = row
    return row


def get_bracket_state(wc_df=None, bracket=None):
    """Estado de cada casilla del cuadro: equipo confirmado o descripción condicional."""
    if wc_df is None:
        wc_df = pd.read_csv(CSV_PATH)
    if bracket is None:
        bracket = json.load(open(BRACKET_JSON_PATH, encoding="utf-8"))

    standings = compute_group_standings(wc_df)
    thirds_ranked = rank_third_place(standings)
    qualified_thirds = (set(thirds_ranked.head(8).group) if len(thirds_ranked) >= 8
                        else set(bracket["qualified_third_place_groups_2026"]))

    rows_by_num = {}
    for m in bracket["round_of_32"]:
        home = _resolve_slot(m["home_slot"], standings, qualified_thirds)
        away = _resolve_slot(m["away_slot"], standings, qualified_thirds)
        _settle_match(wc_df, m["match_number"], "Round of 32",
                     m["home_slot"], m["away_slot"], home, away, rows_by_num)

    for m in bracket["progression"]:
        home = _resolve_source(m["home_source"], rows_by_num)
        away = _resolve_source(m["away_source"], rows_by_num)
        _settle_match(wc_df, m["match_number"], m["round"],
                     m["home_source"], m["away_source"], home, away, rows_by_num)

    cols = ["match_number", "round", "home_team", "away_team", "confirmed_teams",
           "status", "result", "confirmed_by_data", "discrepancy"]
    return pd.DataFrame(rows_by_num.values())[cols].sort_values("match_number").reset_index(drop=True)


standings = compute_group_standings(wc)
thirds_ranked = rank_third_place(standings)
qualified_thirds = set(thirds_ranked.head(8).group) if len(thirds_ranked) >= 8 else set()
print("Grupos completos:", standings.group[standings.group_complete].nunique(), "/ 12")
print("Coincide con bracket_structure.json:",
      qualified_thirds == set(bracket_structure["qualified_third_place_groups_2026"]))

bracket_state = get_bracket_state(wc)
print(f"\nCasillas totales: {len(bracket_state)} | con equipos confirmados: "
      f"{int(bracket_state.confirmed_teams.sum())} | condicionales: "
      f"{int((~bracket_state.confirmed_teams).sum())}")
conflicts = bracket_state[bracket_state.discrepancy.notna()]
if len(conflicts):
    print("\n--- DISCREPANCIAS FBref / Kaggle detectadas ---")
    print(conflicts[["match_number", "round", "home_team", "away_team", "status"]].to_string(index=False))

Grupos completos: 12 / 12
Coincide con bracket_structure.json: True

Casillas totales: 32 | con equipos confirmados: 28 | condicionales: 4


## 10. Peso de la competición — `competition_weight`

`pre_world_cup_form.csv` mezcla amistosos, clasificación y repescas; no todos
los partidos previos al Mundial dicen lo mismo sobre el nivel de una selección.
Añadimos un peso de importancia para usarlo tanto en las medias de forma como,
en el notebook de modelo, como `sample_weight` de XGBoost.

In [16]:
UNKNOWN_COMPETITIONS = set()

# orden de reglas: la primera que caza gana (así "World Cup Qualifiers" no cae
# en la regla genérica de "World Cup" = Mundial)
COMPETITION_WEIGHT_RULES = [
    (re.compile(r"friendly|friendlies", re.I), 1.0, "Amistoso"),
    (re.compile(r"qualif.*play.?off|play.?off.*qualif|intercontinental|repesca", re.I),
     3.0, "Repesca / play-off intercontinental"),
    (re.compile(r"qualif", re.I), 2.5, "Clasificación mundial"),
    (re.compile(r"\bworld cup\b", re.I), 4.0, "Mundial"),
    (re.compile(r"(final|semi-?final).*(nations league|copa am[eé]rica|gold cup|"
                r"afcon|african cup|asian cup|euro)", re.I),
     3.0, "Semifinal/final de copa continental"),
    (re.compile(r"nations league|copa am[eé]rica|gold cup|afcon|african cup of nations|"
                r"asian cup|european championship", re.I),
     2.0, "Copa continental / Nations League (fase de grupos)"),
]


def normalize_competition_weight(competition_name):
    """Texto libre de `competition` -> peso de importancia (1.0 amistoso .. 4.0 Mundial).

    Si no encaja con ninguna regla conocida, lo loguea en UNKNOWN_COMPETITIONS
    y devuelve None en vez de asignar un peso por defecto en silencio.
    """
    if not isinstance(competition_name, str) or not competition_name.strip():
        UNKNOWN_COMPETITIONS.add(competition_name)
        print("AVISO: competición vacía -> sin peso asignado")
        return None
    for pattern, weight, _label in COMPETITION_WEIGHT_RULES:
        if pattern.search(competition_name):
            return weight
    UNKNOWN_COMPETITIONS.add(competition_name)
    print(f"AVISO: competición no reconocida, revisar manualmente: {competition_name!r}")
    return None


form["competition_weight"] = form["competition"].apply(normalize_competition_weight)
wc["competition"] = "FIFA World Cup"
wc["competition_weight"] = wc["competition"].apply(normalize_competition_weight)

form.to_csv(FORM_CSV_PATH, index=False)
wc.to_csv(CSV_PATH, index=False)

print("Competiciones encontradas en pre_world_cup_form.csv y su peso asignado:")
print(form[["competition", "competition_weight"]].drop_duplicates()
      .sort_values("competition_weight").to_string(index=False))
print(f"\nSin reconocer: {UNKNOWN_COMPETITIONS or 'ninguna'}")

Competiciones encontradas en pre_world_cup_form.csv y su peso asignado:
                     competition  competition_weight
        International Friendlies                 1.0
                        Gold Cup                 2.0
   World Cup Qualifiers (Europe)                 2.5
World Cup Qualification Playoffs                 3.0

Sin reconocer: ninguna


## 11. Resumen final — `data/` listo para `mundial2026_modelo.ipynb`

Exportamos también `data/teams.csv` (Elo, ranking FIFA y grupo) para que el
notebook de modelo no necesite llamar a `kagglehub` — así queda garantizado
que no hace ninguna petición de red.

In [17]:
kaggle["teams"].to_csv(TEAMS_CSV_PATH, index=False)

print("=== world_cup_matches.csv ===")
print(f"Total: {len(wc)} | jugados: {(wc['status'] == 'played').sum()} "
      f"| futuros: {(wc['status'] == 'scheduled').sum()}")
print(f"Con xG: {int(wc['home_xg'].notna().sum())} | con stats: {int(wc['stats_scraped'].sum())}")
print("stats_source:", wc["stats_source"].value_counts(dropna=False).to_dict())

print("\n=== pre_world_cup_form.csv ===")
print(f"Filas: {len(form)} | selecciones: {form['team'].nunique()}")

print("\n=== bracket_structure.json ===")
print(f"{len(bracket_structure['round_of_32'])} partidos de dieciseisavos + "
      f"{len(bracket_structure['progression'])} de progresión")

print("\n=== teams.csv ===")
print(f"Guardado {TEAMS_CSV_PATH} ({len(kaggle['teams'])} selecciones)")

print("\nTodo listo en data/ para mundial2026_modelo.ipynb.")

=== world_cup_matches.csv ===
Total: 112 | jugados: 96 | futuros: 16
Con xG: 96 | con stats: 96
stats_source: {'kaggle+fbref': 96, nan: 16}

=== pre_world_cup_form.csv ===
Filas: 240 | selecciones: 48

=== bracket_structure.json ===
16 partidos de dieciseisavos + 16 de progresión

=== teams.csv ===
Guardado data\teams.csv (48 selecciones)

Todo listo en data/ para mundial2026_modelo.ipynb.
